In [ ]:
# url, material, color

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from deep_translator import GoogleTranslator
import pandas as pd, time, random

# ==============================
# ⚙️ 설정
# ==============================
RANKING_URL = "https://www.musinsa.com/main/musinsa/ranking?gf=A&storeCode=musinsa&sectionId=199&contentsId=&categoryCode=003000&ageBand=AGE_BAND_ALL&subPan=product&period=MONTHLY"
TARGET_URLS = 1000
PAGE_LOAD = 4.0
NOTICE_WAIT = 2.0
RETRY_SLEEP = 0.8
SAVE_INTERVAL = 50

# ==============================
# 🌐 드라이버 설정
# ==============================
options = Options()
options.add_argument("--window-size=1400,900")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
# options.add_argument("--headless")  # 필요시 활성화
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

# ==============================
# 🌍 번역기 설정
# ==============================
translator = GoogleTranslator(source="ko", target="en")

def translate_auto(text):
    if not text:
        return None
    try:
        return translator.translate(text)
    except:
        return text  # 번역 실패 시 원문 유지

# ==============================
# 1️⃣ URL 무한 스크롤 수집
# ==============================
driver.get(RANKING_URL)
time.sleep(7)

collected = set()
last_count, stable_loops = 0, 0

def current_links():
    hrefs = driver.execute_script("""
        return Array.from(document.querySelectorAll('a[href*="/products/"]'))
                     .map(a => a.href.split('?')[0]);
    """)
    return [h for h in hrefs if "/products/" in h]

print("📜 상품 URL 수집 시작...")

while len(collected) < TARGET_URLS and stable_loops < 15:
    for _ in range(10):
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.9);")
        time.sleep(random.uniform(0.8, 1.2))
        for h in current_links():
            collected.add(h)

    # 성장 감시
    if len(collected) > last_count:
        stable_loops = 0
        last_count = len(collected)
        print(f"📦 현재 {len(collected)}개 수집됨")
    else:
        stable_loops += 1
        print(f"⏳ 로딩 정체 감지 ({stable_loops}/15)")

    if stable_loops >= 15:
        print("⚠️ 스크롤 정체 — 중단")
        break

product_urls = list(dict.fromkeys(collected))[:TARGET_URLS]
print(f"\n✅ URL 수집 완료: {len(product_urls)}개\n")

# ==============================
# 2️⃣ 상품 상세정보 함수
# ==============================
NOTICE_BTN = (
    "#root > div.Layout__Container-sc-3weaze-0.cbLSDw "
    "> div.VariableArea__Container-sc-4n9q35-0.bliLNC "
    "> div.UILayout__Wrap-sc-18f7p3t-0.SwBfi.gtm-impression-content "
    "> div:nth-child(2) > div:nth-child(2)"
)

def open_notice():
    """상품 고시정보 버튼 클릭 (안정화 버전)"""
    try:
        btn = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, NOTICE_BTN))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        time.sleep(0.8)

        # Dimmed 제거 대기
        try:
            WebDriverWait(driver, 5).until_not(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.Dimmed-sc-53e7ju-0"))
            )
        except TimeoutException:
            pass

        # 클릭 시도
        for attempt in range(3):
            try:
                ActionChains(driver).move_to_element(btn).click().perform()
                break
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except Exception:
                time.sleep(1.5)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div[id^='radix-']"))
        )
        time.sleep(NOTICE_WAIT)
        return True
    except:
        return False

def get_material_color_via_css(max_retry=5, sleep_sec=RETRY_SLEEP):
    mat_sel = "div[id^='radix-'] > div > div > dl > dd:nth-child(2)"
    col_sel = "div[id^='radix-'] > div > div > dl > dd:nth-child(4)"
    material, color = None, None
    for _ in range(max_retry):
        material = driver.execute_script("var n=document.querySelector(arguments[0]); return n? n.textContent.trim(): null;", mat_sel)
        color = driver.execute_script("var n=document.querySelector(arguments[0]); return n? n.textContent.trim(): null;", col_sel)
        if material or color:
            break
        time.sleep(sleep_sec)
    return material, color

# ==============================
# 3️⃣ 크롤링 루프 (처음부터)
# ==============================
rows = []

for idx, url in enumerate(product_urls, start=1):
    success = False
    for attempt in range(3):
        try:
            driver.get(url)
            time.sleep(PAGE_LOAD)
            driver.execute_script("window.scrollBy(0, document.body.scrollHeight * 0.5);")
            time.sleep(1.5)

            material, color = None, None
            if open_notice():
                material, color = get_material_color_via_css()

            rows.append({
                "url": url,
                "material_en": translate_auto(material),
                "color_en": translate_auto(color)
            })

            print(f"✅ {idx}/{len(product_urls)} 완료")
            success = True
            break
        except Exception as e:
            print(f"⚠️ {idx}번째 실패 ({attempt+1}/3): {e}")
            time.sleep(3)

    if not success:
        rows.append({"url": url, "material_en": None, "color_en": None})
        print("🚫 최종 실패 — 건너뜀")

    if idx % SAVE_INTERVAL == 0:
        pd.DataFrame(rows).to_excel("C03_savefile_bottom_v1.xlsx", index=False)
        print(f"💾 {idx}개 수집 완료 — 중간 저장")

# ==============================
# 4️⃣ 최종 저장
# ==============================
driver.quit()
pd.DataFrame(rows).to_excel("C03_1000_bottom_v1.xlsx", index=False)
print("\n🎯 저장 완료 → C03_1000_bottom_v1.xlsx (1000개 안정 수집 완료)")

📜 상품 URL 수집 시작...
📦 현재 113개 수집됨
📦 현재 221개 수집됨
📦 현재 311개 수집됨
📦 현재 419개 수집됨
📦 현재 509개 수집됨
📦 현재 617개 수집됨
📦 현재 707개 수집됨
📦 현재 815개 수집됨
📦 현재 905개 수집됨
📦 현재 1013개 수집됨

✅ URL 수집 완료: 1000개

✅ 1/1000 완료
✅ 2/1000 완료
✅ 3/1000 완료
✅ 4/1000 완료
✅ 5/1000 완료
✅ 6/1000 완료
✅ 7/1000 완료
✅ 8/1000 완료
✅ 9/1000 완료
✅ 10/1000 완료
✅ 11/1000 완료
✅ 12/1000 완료
✅ 13/1000 완료
✅ 14/1000 완료
✅ 15/1000 완료
✅ 16/1000 완료
✅ 17/1000 완료
✅ 18/1000 완료
✅ 19/1000 완료
✅ 20/1000 완료
✅ 21/1000 완료
✅ 22/1000 완료
✅ 23/1000 완료
✅ 24/1000 완료
✅ 25/1000 완료
✅ 26/1000 완료
✅ 27/1000 완료
✅ 28/1000 완료
✅ 29/1000 완료
✅ 30/1000 완료
✅ 31/1000 완료
✅ 32/1000 완료
✅ 33/1000 완료
✅ 34/1000 완료
✅ 35/1000 완료
✅ 36/1000 완료
✅ 37/1000 완료
✅ 38/1000 완료
✅ 39/1000 완료
✅ 40/1000 완료
✅ 41/1000 완료
✅ 42/1000 완료
✅ 43/1000 완료
✅ 44/1000 완료
✅ 45/1000 완료
✅ 46/1000 완료
✅ 47/1000 완료
✅ 48/1000 완료
✅ 49/1000 완료
✅ 50/1000 완료
💾 50개 수집 완료 — 중간 저장
✅ 51/1000 완료
✅ 52/1000 완료
✅ 53/1000 완료
✅ 54/1000 완료
✅ 55/1000 완료
✅ 56/1000 완료
✅ 57/1000 완료
✅ 58/1000 완료
✅ 59/1000 완료
✅ 60/1000 완료
✅ 61/1000 완료
✅ 62/1000 완료
✅ 6